In [4]:
#!pip install sentence_transformers
#!pip install ag2[retrievechat]
#!pip install pyautogen==0.7.2
#https://microsoft.github.io/autogen/0.2/docs/notebooks/agentchat_RetrieveChat/

# Setting base for Retrive Chat

In [5]:
import json
import os

import chromadb

import autogen
from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Accepted file formats for that can be stored in
# a vector database instance
from autogen.retrieve_utils import TEXT_FORMATS

config_list = autogen.config_list_from_json("OAI_CONFIG_LIST.json")

assert len(config_list) > 0
print("models to use: ", [config_list[i]["model"] for i in range(len(config_list))])

Patching name='__init__', member=<function LLMLingua.__init__ at 0x0000020DA3385620>, patched=<function function.__call__ at 0x0000020DA3385580>
Patching name='compress_text', member=<function LLMLingua.compress_text at 0x0000020DA33856C0>, patched=<function function.__call__ at 0x0000020DA33858A0>
models to use:  ['gpt-35-turbo', 'gpt-4-vision-preview', 'dalle', 'gemini-pro', 'gemini-1.5-pro-001', 'gemini-1.5-pro', 'gemini-pro-vision']


In [6]:
print("Accepted file formats for `docs_path`:")
print(TEXT_FORMATS)

Accepted file formats for `docs_path`:
['txt', 'json', 'csv', 'tsv', 'md', 'html', 'htm', 'rtf', 'rst', 'jsonl', 'log', 'xml', 'yaml', 'yml', 'pdf', 'mdx']


In [7]:
config_list = [
    {
        "model": "models/gemini-2.5-flash",
        "api_key": "",
        "api_type": "google"
    }
]

In [8]:
import agentops
agentops.init("")

🖇 AgentOps: [ConcurrentFuturesInstrumentor] Successfully instrumented concurrent.futures.ThreadPoolExecutor


In [9]:
# 1. create an AssistantAgent instance named "assistant"
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config={
        "timeout": 600,
        "cache_seed": 42,
        "config_list": config_list,
    },
)


In [10]:
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Extract text from local PDF files
pdf_files = ["book.pdf"]
#docs_content = [extract_text_from_pdf(file) for file in pdf_files]

from autogen.retrieve_utils import TEXT_FORMATS,extract_text_from_pdf
docs_content= extract_text_from_pdf("book.pdf")

In [11]:
# Define the RetrieveUserProxyAgent instance
ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "qa",
        "docs_path": ["autogen-docs/book.pdf"],
        "docs_content": docs_content,  # Provide the extracted content directly
        "chunk_token_size": 200,
        "model": config_list[0]["model"],
        "vector_db": "chroma",
        "overwrite": True,
        "get_or_create": True,
    },
    code_execution_config=False,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\PAVAN\anaconda3\envs\AIagent\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PAVAN\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Process Starts for Scenario 1 : Generate code based off docstrings w/o human feedback

In [12]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "summarize the content and use python code to any calculation given in the book"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)
#agentops.end_session("Success")

Trying to create collection.


max_tokens is too small to fit a single line of text. Breaking this line:
	The 22 ...
Failed to split docs with must_break_at_empty_line being True, set to False.
2026-04-14 22:47:37,013 - autogen.agentchat.contrib.retrieve_user_proxy_agent - INFO - Found 215 chunks.
2026-04-14 22:47:37,086 - autogen.agentchat.contrib.vectordb.chromadb - INFO - No content embedding is provided. Will use the VectorDB's embedding function to generate the content embedding.
Model models/gemini-2.5-flash not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['fe1105cf', '8bf7f8ee', '1d0894aa', '839a2e04', 'd11d708f', '150c596a', 'fbda2b4c', '59e93e46', '1613971f', '25edaaee', '7091167f', 'c1c95a95', '43a28a91', '16a1e2eb', 'e7d73411', 'a5e9b671', 'a1061522', '05dac356', '1097b211', 'd41e75dd']]
Adding content of doc fe1105cf to context.


Model models/gemini-2.5-flash not found. Using cl100k_base encoding.


Adding content of doc 8bf7f8ee to context.


...

Model models/gemini-2.5-flash not found. Using cl100k_base encoding.


Adding content of doc 1097b211 to context.


Model models/gemini-2.5-flash not found. Using cl100k_base encoding.


Adding content of doc d41e75dd to context.
ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: summarize the content and use python code to any calculation given in the book

Context is: hottest subject in the business management field is
benchmarking. Touted as the “ultimate competitive
strategy,” benchmarking is the process of comparing
and evaluating your company’ s products against the
best in the industry. It’ s an essential element in a
process often called “total quality management.”
Unfortunately, benchmarking doesn’t work.
Regardless of reality, people perceive the first product
into the mind as superior. Marketing is a battle of per-
ceptions, not products.
So what’ s the name of the first b

🖇 AgentOps: [agentops.InternalSpanProcessor] Error uploading logfile: Upload failed: 401


In [ ]:
#agentops.end_session("Success")

# Process Starts for Scenario 2 : Answer a question based off docstrings w/o human feedback

In [13]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "You are a creative head . do many brainstorms"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)

Model models/gemini-2.5-flash not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['1097b211', '7e1ac0cd', '0e08a976', '81d8712b', 'be843ac3', 'a5e9b671', 'b2f1dbea', 'ba9a27e8', '21ec236c', '8126c05a', 'cf32340a', '7e3514a1', '89fd1dac', '5b19ade7', '10e789b5', '0e403d83', '71a3bd57', '1e7668a9', 'a1f246ff', '4cf44936']]
Adding content of doc 1097b211 to context.


Model models/gemini-2.5-flash not found. Using cl100k_base encoding.


Adding content of doc 7e1ac0cd to context.


Model models/gemini-2.5-flash not found. Using cl100k_base encoding.


...

Adding content of doc a1f246ff to context.


Model models/gemini-2.5-flash not found. Using cl100k_base encoding.


Adding content of doc 4cf44936 to context.
ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: You are a creative head . do many brainstorms

Context is: Y our marketing strategy should depend on how
soon you got into the mind and consequently which
rung of the ladder you occupy. The higher the better, of
course.
Take Avis, for example. For years the company
advertised the high quality of its rent-a-car service.
“Finest in rent-a-cars” was one of its campaigns. The
reader looked at the ad and wondered, How could they
have the finest rent-a-car service when they’re not on
the top rung of my ladder?
Then Avis did the one thing you have to do to make
progress inside the mind of the prospect. They
acknowl

🖇 AgentOps: Error in span models/gemini-2.5-flash.llm: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 27.429416935s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 27
}
]
🖇 AgentOps: Error in span ag2.agent.assistant.receive

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 27.429416935s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 27
}
]

# Process Starts for Scenario 3 : Generate code based off docstrings w/ human feedback

# Process Starts for Scenario 4: Answer a question based off docstrings w/ human feedback

In [15]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
qa_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. "
chat_result = ragproxyagent.initiate_chat(
    assistant, message=ragproxyagent.message_generator, problem=qa_problem
)  # type "exit" to exit the conversation

Model models/gemini-2.5-flash not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['1097b211', '81d8712b', '71a3bd57', '89fd1dac', 'be843ac3', '150c596a', 'b210f301', '8a2a101c', 'ba9a27e8', '0e08a976', '0e403d83', '788184ee', '039e8163', 'a1f246ff', 'd5a791b1', '08e00728', '083871f5', 'c1b21496', '0c4927c1', '8cc896cd']]
Adding content of doc 1097b211 to context.


Model models/gemini-2.5-flash not found. Using cl100k_base encoding.


Adding content of doc 81d8712b to context.


Model models/gemini-2.5-flash not found. Using cl100k_base encoding.


...

Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Can you reshare here?


ragproxyagent (to assistant):

Can you reshare here?

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

Here are the 5 marketing ideas I previously shared:

1.  **Lead a Niche**: Aim to be perceived as the *first* or *leading* institute for a specific AI/DS specialty (e.g., "First for MLOps").
2.  **Shape Perception**: Focus marketing on crafting a specific, powerful perception in prospects' minds (e.g., "The most practical training").
3.  **Own a Word**: Concentrate messaging to own a single, simple word or concept (e.g., "Job-ready," "Innovation," "Results").
4.  **Acknowledge Position**: If not the leader, highlight how you "try harder" in a specific area (e.g., "Smaller classes, personalized mentorship").
5.  **Secure Resources**: Ensure adequate funding for sustained marketing to drive ideas into the mind.

--------------------------------------------------------------------------------


Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Thank you exit 


ragproxyagent (to assistant):

Thank you exit 

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

You're welcome! Goodbye.

--------------------------------------------------------------------------------


Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  exit
